In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# path constants
DATA = '../data/individual/processed'
PSY = '../data/individual/psychometric'

# load eye data
baseline_eye_tracking = pd.read_csv(f'{DATA}/sed.csv')
eye_tracking_01 = pd.read_csv(f'{DATA}/sed_01.csv')
eye_tracking_02 = pd.read_csv(f'{DATA}/sed_02.csv')
eye_tracking_03 = pd.read_csv(f'{DATA}/sed_03.csv')

# load psychometric data
psychometric_01 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_01.csv')
psychometric_02 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_02.csv')
psychometric_03 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_03.csv')

# rename columns
columns_mapping = {
    'datetime': 'timestamp',
    'pupil': 'pupil_dilation',
    'leftEyeOpen': 'left_blink',
    'rightEyeOpen': 'right_blink'
}
baseline_eye_tracking = baseline_eye_tracking.rename(columns=columns_mapping)
eye_tracking_01 = eye_tracking_01.rename(columns=columns_mapping)
eye_tracking_02 = eye_tracking_02.rename(columns=columns_mapping)
eye_tracking_03 = eye_tracking_03.rename(columns=columns_mapping)

# clean eye data
def clean_eye_tracking_data(eye_tracking_data):
    eye_tracking_data['timestamp'] = pd.to_datetime(
        eye_tracking_data['timestamp'], utc=True, errors='coerce'
    ).dt.tz_convert(None)
    eye_tracking_data = eye_tracking_data.dropna(subset=['timestamp'])
    return eye_tracking_data

baseline_eye_tracking = clean_eye_tracking_data(baseline_eye_tracking)
eye_tracking_01 = clean_eye_tracking_data(eye_tracking_01)
eye_tracking_02 = clean_eye_tracking_data(eye_tracking_02)
eye_tracking_03 = clean_eye_tracking_data(eye_tracking_03)

# clean psychometric datetimes
for psy_df in [psychometric_01, psychometric_02, psychometric_03]:
    psy_df['Question Start Time'] = pd.to_datetime(
        psy_df['Question Start Time'], utc=True, errors='coerce'
    ).dt.tz_convert(None)
    psy_df['Question Answer Time'] = pd.to_datetime(
        psy_df['Question Answer Time'], utc=True, errors='coerce'
    ).dt.tz_convert(None)
psychometric_01 = psychometric_01.dropna(subset=['Question Start Time'])
psychometric_02 = psychometric_02.dropna(subset=['Question Start Time'])
psychometric_03 = psychometric_03.dropna(subset=['Question Start Time'])

# filter by time range
def filter_eye_tracking_data(eye_tracking_data, questions):
    start_time = questions['Question Start Time'].min()
    end_time = questions['Question Answer Time'].max()
    return eye_tracking_data[
        (eye_tracking_data['timestamp'] >= start_time) &
        (eye_tracking_data['timestamp'] <= end_time)
    ]

# calculate eye metrics
def calculate_eye_tracking_metrics(eye_tracking_data):
    average_pupil_dilation = eye_tracking_data['pupil_dilation'].mean()
    blink_detection_threshold = 1.0
    left_blink_count = (
        (eye_tracking_data['left_blink'] > blink_detection_threshold) &
        (eye_tracking_data['left_blink'].shift(-1) <= blink_detection_threshold)
    ).sum()
    right_blink_count = (
        (eye_tracking_data['right_blink'] > blink_detection_threshold) &
        (eye_tracking_data['right_blink'].shift(-1) <= blink_detection_threshold)
    ).sum()
    total_duration_minutes = (
        eye_tracking_data['timestamp'].max() - eye_tracking_data['timestamp'].min()
    ).total_seconds() / 60
    left_blink_rate = left_blink_count / total_duration_minutes
    right_blink_rate = right_blink_count / total_duration_minutes
    return average_pupil_dilation, left_blink_rate, right_blink_rate

# detect baseline increase
def detect_significant_increase(test_metrics, baseline_metrics):
    pupil_dilation_increase = test_metrics[0] > baseline_metrics[0]
    left_blink_rate_increase = test_metrics[1] > baseline_metrics[1]
    right_blink_rate_increase = test_metrics[2] > baseline_metrics[2]
    return pupil_dilation_increase, left_blink_rate_increase, right_blink_rate_increase

# baseline metrics
baseline_metrics = calculate_eye_tracking_metrics(baseline_eye_tracking)

# question type constants
question_types = ['HADS', 'STAI-S', 'STAI-T', 'BFI', 'FQ']

print("Setup complete. All data loaded and cleaned.")

In [ ]:
# --- HADS-only analysis with bar chart ---

# Filter for HADS questions
questions_hads_01 = psychometric_01[psychometric_01['Type'] == 'HADS'].copy()
questions_hads_02 = psychometric_02[psychometric_02['Type'] == 'HADS'].copy()
questions_hads_03 = psychometric_03[psychometric_03['Type'] == 'HADS'].copy()

eye_tracking_hads_01 = filter_eye_tracking_data(eye_tracking_01, questions_hads_01)
eye_tracking_hads_02 = filter_eye_tracking_data(eye_tracking_02, questions_hads_02)
eye_tracking_hads_03 = filter_eye_tracking_data(eye_tracking_03, questions_hads_03)

# Calculate metrics
metrics_01 = calculate_eye_tracking_metrics(eye_tracking_hads_01)
metrics_02 = calculate_eye_tracking_metrics(eye_tracking_hads_02)
metrics_03 = calculate_eye_tracking_metrics(eye_tracking_hads_03)

# Compare against baseline
significant_increase_01 = detect_significant_increase(metrics_01, baseline_metrics)
significant_increase_02 = detect_significant_increase(metrics_02, baseline_metrics)
significant_increase_03 = detect_significant_increase(metrics_03, baseline_metrics)

# Create results table
results = pd.DataFrame({
    'Test': ['Baseline', 'Test 01', 'Test 02', 'Test 03'],
    'Average Pupil Dilation': [baseline_metrics[0], metrics_01[0], metrics_02[0], metrics_03[0]],
    'Left Blink Rate (blinks/min)': [baseline_metrics[1], metrics_01[1], metrics_02[1], metrics_03[1]],
    'Right Blink Rate (blinks/min)': [baseline_metrics[2], metrics_01[2], metrics_02[2], metrics_03[2]],
    'Pupil Dilation Increase': ['-', significant_increase_01[0], significant_increase_02[0], significant_increase_03[0]],
    'Left Blink Rate Increase': ['-', significant_increase_01[1], significant_increase_02[1], significant_increase_03[1]],
    'Right Blink Rate Increase': ['-', significant_increase_01[2], significant_increase_02[2], significant_increase_03[2]]
})

# Display
print("Eye-Tracking Metrics for HADS:")
print(results.to_string(index=False))

# Visualize the results
results_plot = results.set_index('Test')
results_plot[['Average Pupil Dilation', 'Left Blink Rate (blinks/min)', 'Right Blink Rate (blinks/min)']].plot(kind='bar', figsize=(10, 6))
plt.title('Eye-Tracking Metrics for HADS')
plt.ylabel('Metrics')
plt.show()
plt.close()

In [ ]:
# --- HADS-only analysis with text output ---

# Filter for HADS questions
questions_hads_01 = psychometric_01[psychometric_01['Type'] == 'HADS'].copy()
questions_hads_02 = psychometric_02[psychometric_02['Type'] == 'HADS'].copy()
questions_hads_03 = psychometric_03[psychometric_03['Type'] == 'HADS'].copy()

eye_tracking_hads_01 = filter_eye_tracking_data(eye_tracking_01, questions_hads_01)
eye_tracking_hads_02 = filter_eye_tracking_data(eye_tracking_02, questions_hads_02)
eye_tracking_hads_03 = filter_eye_tracking_data(eye_tracking_03, questions_hads_03)

# Calculate metrics
metrics_01 = calculate_eye_tracking_metrics(eye_tracking_hads_01)
metrics_02 = calculate_eye_tracking_metrics(eye_tracking_hads_02)
metrics_03 = calculate_eye_tracking_metrics(eye_tracking_hads_03)

# Compare against baseline
significant_increase_01 = detect_significant_increase(metrics_01, baseline_metrics)
significant_increase_02 = detect_significant_increase(metrics_02, baseline_metrics)
significant_increase_03 = detect_significant_increase(metrics_03, baseline_metrics)

# Display the results
print(f"Baseline Metrics:\n  Average Pupil Dilation: {baseline_metrics[0]:.2f}\n  Left Blink Rate: {baseline_metrics[1]:.2f} blinks/min\n  Right Blink Rate: {baseline_metrics[2]:.2f} blinks/min\n")

print("Test 01:")
print(f"  Average Pupil Dilation: {metrics_01[0]:.2f}")
print(f"  Left Blink Rate: {metrics_01[1]:.2f} blinks/min")
print(f"  Right Blink Rate: {metrics_01[2]:.2f} blinks/min")
print(f"  Significant Pupil Dilation Increase: {'Yes' if significant_increase_01[0] else 'No'}")
print(f"  Significant Left Blink Rate Increase: {'Yes' if significant_increase_01[1] else 'No'}")
print(f"  Significant Right Blink Rate Increase: {'Yes' if significant_increase_01[2] else 'No'}\n")

print("Test 02:")
print(f"  Average Pupil Dilation: {metrics_02[0]:.2f}")
print(f"  Left Blink Rate: {metrics_02[1]:.2f} blinks/min")
print(f"  Right Blink Rate: {metrics_02[2]:.2f} blinks/min")
print(f"  Significant Pupil Dilation Increase: {'Yes' if significant_increase_02[0] else 'No'}")
print(f"  Significant Left Blink Rate Increase: {'Yes' if significant_increase_02[1] else 'No'}")
print(f"  Significant Right Blink Rate Increase: {'Yes' if significant_increase_02[2] else 'No'}\n")

print("Test 03:")
print(f"  Average Pupil Dilation: {metrics_03[0]:.2f}")
print(f"  Left Blink Rate: {metrics_03[1]:.2f} blinks/min")
print(f"  Right Blink Rate: {metrics_03[2]:.2f} blinks/min")
print(f"  Significant Pupil Dilation Increase: {'Yes' if significant_increase_03[0] else 'No'}")
print(f"  Significant Left Blink Rate Increase: {'Yes' if significant_increase_03[1] else 'No'}")
print(f"  Significant Right Blink Rate Increase: {'Yes' if significant_increase_03[2] else 'No'}\n")

In [ ]:
# --- HADS-only analysis with time ranges ---

# Filter for HADS questions
questions_hads_01 = psychometric_01[psychometric_01['Type'] == 'HADS'].copy()
questions_hads_02 = psychometric_02[psychometric_02['Type'] == 'HADS'].copy()
questions_hads_03 = psychometric_03[psychometric_03['Type'] == 'HADS'].copy()

eye_tracking_hads_01 = filter_eye_tracking_data(eye_tracking_01, questions_hads_01)
eye_tracking_hads_02 = filter_eye_tracking_data(eye_tracking_02, questions_hads_02)
eye_tracking_hads_03 = filter_eye_tracking_data(eye_tracking_03, questions_hads_03)

# Calculate metrics
metrics_01 = calculate_eye_tracking_metrics(eye_tracking_hads_01)
metrics_02 = calculate_eye_tracking_metrics(eye_tracking_hads_02)
metrics_03 = calculate_eye_tracking_metrics(eye_tracking_hads_03)

# Compare against baseline
significant_increase_01 = detect_significant_increase(metrics_01, baseline_metrics)
significant_increase_02 = detect_significant_increase(metrics_02, baseline_metrics)
significant_increase_03 = detect_significant_increase(metrics_03, baseline_metrics)

# Display results with time ranges
print(f"Baseline Metrics:\n  Average Pupil Dilation: {baseline_metrics[0]:.2f}\n  Left Blink Rate: {baseline_metrics[1]:.2f} blinks/min\n  Right Blink Rate: {baseline_metrics[2]:.2f} blinks/min\n")

print("Test 01 (HADS):")
print(f"  Start time for HADS: {questions_hads_01['Question Start Time'].min().strftime('%H:%M:%S')}")
print(f"  End time for HADS: {questions_hads_01['Question Answer Time'].max().strftime('%H:%M:%S')}")
print(f"  Average Pupil Dilation: {metrics_01[0]:.2f}")
print(f"  Left Blink Rate: {metrics_01[1]:.2f} blinks/min")
print(f"  Right Blink Rate: {metrics_01[2]:.2f} blinks/min")
print(f"  Significant Increase in Pupil Dilation: {'Yes' if significant_increase_01[0] else 'No'}")
print(f"  Significant Increase in Left Blink Rate: {'Yes' if significant_increase_01[1] else 'No'}")
print(f"  Significant Increase in Right Blink Rate: {'Yes' if significant_increase_01[2] else 'No'}\n")

print("Test 02 (HADS):")
print(f"  Start time for HADS: {questions_hads_02['Question Start Time'].min().strftime('%H:%M:%S')}")
print(f"  End time for HADS: {questions_hads_02['Question Answer Time'].max().strftime('%H:%M:%S')}")
print(f"  Average Pupil Dilation: {metrics_02[0]:.2f}")
print(f"  Left Blink Rate: {metrics_02[1]:.2f} blinks/min")
print(f"  Right Blink Rate: {metrics_02[2]:.2f} blinks/min")
print(f"  Significant Increase in Pupil Dilation: {'Yes' if significant_increase_02[0] else 'No'}")
print(f"  Significant Increase in Left Blink Rate: {'Yes' if significant_increase_02[1] else 'No'}")
print(f"  Significant Increase in Right Blink Rate: {'Yes' if significant_increase_02[2] else 'No'}\n")

print("Test 03 (HADS):")
print(f"  Start time for HADS: {questions_hads_03['Question Start Time'].min().strftime('%H:%M:%S')}")
print(f"  End time for HADS: {questions_hads_03['Question Answer Time'].max().strftime('%H:%M:%S')}")
print(f"  Average Pupil Dilation: {metrics_03[0]:.2f}")
print(f"  Left Blink Rate: {metrics_03[1]:.2f} blinks/min")
print(f"  Right Blink Rate: {metrics_03[2]:.2f} blinks/min")
print(f"  Significant Increase in Pupil Dilation: {'Yes' if significant_increase_03[0] else 'No'}")
print(f"  Significant Increase in Left Blink Rate: {'Yes' if significant_increase_03[1] else 'No'}")
print(f"  Significant Increase in Right Blink Rate: {'Yes' if significant_increase_03[2] else 'No'}\n")

In [ ]:
# --- All question types analysis with text table ---

psychometric_data = {
    qt: [
        psychometric_01[psychometric_01['Type'] == qt],
        psychometric_02[psychometric_02['Type'] == qt],
        psychometric_03[psychometric_03['Type'] == qt]
    ]
    for qt in question_types
}

results = []
for question_type in question_types:
    questions_01, questions_02, questions_03 = psychometric_data[question_type]

    eye_tracking_hads_01 = filter_eye_tracking_data(eye_tracking_01, questions_01)
    eye_tracking_hads_02 = filter_eye_tracking_data(eye_tracking_02, questions_02)
    eye_tracking_hads_03 = filter_eye_tracking_data(eye_tracking_03, questions_03)

    metrics_01 = calculate_eye_tracking_metrics(eye_tracking_hads_01)
    metrics_02 = calculate_eye_tracking_metrics(eye_tracking_hads_02)
    metrics_03 = calculate_eye_tracking_metrics(eye_tracking_hads_03)

    significant_increase_01 = detect_significant_increase(metrics_01, baseline_metrics)
    significant_increase_02 = detect_significant_increase(metrics_02, baseline_metrics)
    significant_increase_03 = detect_significant_increase(metrics_03, baseline_metrics)

    results.append({'Test': 'Test 01', 'Type': question_type, 'Start Time': questions_01['Question Start Time'].min().strftime('%H:%M:%S'), 'End Time': questions_01['Question Answer Time'].max().strftime('%H:%M:%S'),
                    'Average Pupil Dilation': metrics_01[0], 'Left Blink Rate': metrics_01[1], 'Right Blink Rate': metrics_01[2],
                    'Significant Increase (Pupil Dilation)': significant_increase_01[0], 'Significant Increase (Left Blink Rate)': significant_increase_01[1],
                    'Significant Increase (Right Blink Rate)': significant_increase_01[2]})

    results.append({'Test': 'Test 02', 'Type': question_type, 'Start Time': questions_02['Question Start Time'].min().strftime('%H:%M:%S'), 'End Time': questions_02['Question Answer Time'].max().strftime('%H:%M:%S'),
                    'Average Pupil Dilation': metrics_02[0], 'Left Blink Rate': metrics_02[1], 'Right Blink Rate': metrics_02[2],
                    'Significant Increase (Pupil Dilation)': significant_increase_02[0], 'Significant Increase (Left Blink Rate)': significant_increase_02[1],
                    'Significant Increase (Right Blink Rate)': significant_increase_02[2]})

    results.append({'Test': 'Test 03', 'Type': question_type, 'Start Time': questions_03['Question Start Time'].min().strftime('%H:%M:%S'), 'End Time': questions_03['Question Answer Time'].max().strftime('%H:%M:%S'),
                    'Average Pupil Dilation': metrics_03[0], 'Left Blink Rate': metrics_03[1], 'Right Blink Rate': metrics_03[2],
                    'Significant Increase (Pupil Dilation)': significant_increase_03[0], 'Significant Increase (Left Blink Rate)': significant_increase_03[1],
                    'Significant Increase (Right Blink Rate)': significant_increase_03[2]})

# Show results
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# --- All question types analysis with line plots ---

psychometric_data = {
    qt: [
        psychometric_01[psychometric_01['Type'] == qt],
        psychometric_02[psychometric_02['Type'] == qt],
        psychometric_03[psychometric_03['Type'] == qt]
    ]
    for qt in question_types
}

results = []
for question_type in question_types:
    questions_01, questions_02, questions_03 = psychometric_data[question_type]

    eye_tracking_hads_01 = filter_eye_tracking_data(eye_tracking_01, questions_01)
    eye_tracking_hads_02 = filter_eye_tracking_data(eye_tracking_02, questions_02)
    eye_tracking_hads_03 = filter_eye_tracking_data(eye_tracking_03, questions_03)

    metrics_01 = calculate_eye_tracking_metrics(eye_tracking_hads_01)
    metrics_02 = calculate_eye_tracking_metrics(eye_tracking_hads_02)
    metrics_03 = calculate_eye_tracking_metrics(eye_tracking_hads_03)

    significant_increase_01 = detect_significant_increase(metrics_01, baseline_metrics)
    significant_increase_02 = detect_significant_increase(metrics_02, baseline_metrics)
    significant_increase_03 = detect_significant_increase(metrics_03, baseline_metrics)

    results.append({'Test': 'Test 01', 'Type': question_type, 'Start Time': questions_01['Question Start Time'].min().strftime('%H:%M:%S'), 'End Time': questions_01['Question Answer Time'].max().strftime('%H:%M:%S'),
                    'Average Pupil Dilation': metrics_01[0], 'Left Blink Rate': metrics_01[1], 'Right Blink Rate': metrics_01[2],
                    'Significant Increase (Pupil Dilation)': significant_increase_01[0], 'Significant Increase (Left Blink Rate)': significant_increase_01[1],
                    'Significant Increase (Right Blink Rate)': significant_increase_01[2]})

    results.append({'Test': 'Test 02', 'Type': question_type, 'Start Time': questions_02['Question Start Time'].min().strftime('%H:%M:%S'), 'End Time': questions_02['Question Answer Time'].max().strftime('%H:%M:%S'),
                    'Average Pupil Dilation': metrics_02[0], 'Left Blink Rate': metrics_02[1], 'Right Blink Rate': metrics_02[2],
                    'Significant Increase (Pupil Dilation)': significant_increase_02[0], 'Significant Increase (Left Blink Rate)': significant_increase_02[1],
                    'Significant Increase (Right Blink Rate)': significant_increase_02[2]})

    results.append({'Test': 'Test 03', 'Type': question_type, 'Start Time': questions_03['Question Start Time'].min().strftime('%H:%M:%S'), 'End Time': questions_03['Question Answer Time'].max().strftime('%H:%M:%S'),
                    'Average Pupil Dilation': metrics_03[0], 'Left Blink Rate': metrics_03[1], 'Right Blink Rate': metrics_03[2],
                    'Significant Increase (Pupil Dilation)': significant_increase_03[0], 'Significant Increase (Left Blink Rate)': significant_increase_03[1],
                    'Significant Increase (Right Blink Rate)': significant_increase_03[2]})

# Create table
results_df = pd.DataFrame(results)

# Plot average pupil dilation
plt.figure(figsize=(14, 6))
for question_type in question_types:
    subset = results_df[results_df['Type'] == question_type]
    plt.plot(subset['Test'], subset['Average Pupil Dilation'], marker='o', label=question_type)

plt.title('Average Pupil Dilation by Test and Question Type')
plt.xlabel('Test')
plt.ylabel('Average Pupil Dilation')
plt.legend(title='Question Type')
plt.grid(True)
plt.show()
plt.close()

# Plot left blink rate
plt.figure(figsize=(14, 6))
for question_type in question_types:
    subset = results_df[results_df['Type'] == question_type]
    plt.plot(subset['Test'], subset['Left Blink Rate'], marker='o', label=question_type)

plt.title('Left Blink Rate by Test and Question Type')
plt.xlabel('Test')
plt.ylabel('Left Blink Rate (blinks/min)')
plt.legend(title='Question Type')
plt.grid(True)
plt.show()
plt.close()

# Plot right blink rate
plt.figure(figsize=(14, 6))
for question_type in question_types:
    subset = results_df[results_df['Type'] == question_type]
    plt.plot(subset['Test'], subset['Right Blink Rate'], marker='o', label=question_type)

plt.title('Right Blink Rate by Test and Question Type')
plt.xlabel('Test')
plt.ylabel('Right Blink Rate (blinks/min)')
plt.legend(title='Question Type')
plt.grid(True)
plt.show()
plt.close()

In [ ]:
# --- All question types analysis with detailed text output ---

psychometric_data = {
    qt: [
        psychometric_01[psychometric_01['Type'] == qt],
        psychometric_02[psychometric_02['Type'] == qt],
        psychometric_03[psychometric_03['Type'] == qt]
    ]
    for qt in question_types
}

results = []
for question_type in question_types:
    questions_01, questions_02, questions_03 = psychometric_data[question_type]

    eye_tracking_hads_01 = filter_eye_tracking_data(eye_tracking_01, questions_01)
    eye_tracking_hads_02 = filter_eye_tracking_data(eye_tracking_02, questions_02)
    eye_tracking_hads_03 = filter_eye_tracking_data(eye_tracking_03, questions_03)

    metrics_01 = calculate_eye_tracking_metrics(eye_tracking_hads_01)
    metrics_02 = calculate_eye_tracking_metrics(eye_tracking_hads_02)
    metrics_03 = calculate_eye_tracking_metrics(eye_tracking_hads_03)

    significant_increase_01 = detect_significant_increase(metrics_01, baseline_metrics)
    significant_increase_02 = detect_significant_increase(metrics_02, baseline_metrics)
    significant_increase_03 = detect_significant_increase(metrics_03, baseline_metrics)

    results.append({'Test': 'Test 01', 'Type': question_type, 'Start Time': questions_01['Question Start Time'].min().strftime('%H:%M:%S'), 'End Time': questions_01['Question Answer Time'].max().strftime('%H:%M:%S'),
                    'Average Pupil Dilation': metrics_01[0], 'Left Blink Rate': metrics_01[1], 'Right Blink Rate': metrics_01[2],
                    'Significant Increase (Pupil Dilation)': significant_increase_01[0], 'Significant Increase (Left Blink Rate)': significant_increase_01[1],
                    'Significant Increase (Right Blink Rate)': significant_increase_01[2]})

    results.append({'Test': 'Test 02', 'Type': question_type, 'Start Time': questions_02['Question Start Time'].min().strftime('%H:%M:%S'), 'End Time': questions_02['Question Answer Time'].max().strftime('%H:%M:%S'),
                    'Average Pupil Dilation': metrics_02[0], 'Left Blink Rate': metrics_02[1], 'Right Blink Rate': metrics_02[2],
                    'Significant Increase (Pupil Dilation)': significant_increase_02[0], 'Significant Increase (Left Blink Rate)': significant_increase_02[1],
                    'Significant Increase (Right Blink Rate)': significant_increase_02[2]})

    results.append({'Test': 'Test 03', 'Type': question_type, 'Start Time': questions_03['Question Start Time'].min().strftime('%H:%M:%S'), 'End Time': questions_03['Question Answer Time'].max().strftime('%H:%M:%S'),
                    'Average Pupil Dilation': metrics_03[0], 'Left Blink Rate': metrics_03[1], 'Right Blink Rate': metrics_03[2],
                    'Significant Increase (Pupil Dilation)': significant_increase_03[0], 'Significant Increase (Left Blink Rate)': significant_increase_03[1],
                    'Significant Increase (Right Blink Rate)': significant_increase_03[2]})

# Create table
results_df = pd.DataFrame(results)

# Show detailed results per question type
for question_type in question_types:
    print(f"\nResults for {question_type}:\n")
    subset = results_df[results_df['Type'] == question_type]
    for index, row in subset.iterrows():
        print(f"Test {row['Test']}:")
        print(f"  Start Time: {row['Start Time']}")
        print(f"  End Time: {row['End Time']}")
        print(f"  Average Pupil Dilation: {row['Average Pupil Dilation']:.2f}")
        print(f"  Left Blink Rate: {row['Left Blink Rate']:.2f} blinks/min")
        print(f"  Right Blink Rate: {row['Right Blink Rate']:.2f} blinks/min")
        print(f"  Significant Increase (Pupil Dilation): {'Yes' if row['Significant Increase (Pupil Dilation)'] else 'No'}")
        print(f"  Significant Increase (Left Blink Rate): {'Yes' if row['Significant Increase (Left Blink Rate)'] else 'No'}")
        print(f"  Significant Increase (Right Blink Rate): {'Yes' if row['Significant Increase (Right Blink Rate)'] else 'No'}\n")

In [ ]:
# --- All question types analysis (simple metrics, no significance test) ---

psychometric_data = {
    qt: [
        psychometric_01[psychometric_01['Type'] == qt],
        psychometric_02[psychometric_02['Type'] == qt],
        psychometric_03[psychometric_03['Type'] == qt]
    ]
    for qt in question_types
}

results = []
for question_type in question_types:
    questions_01, questions_02, questions_03 = psychometric_data[question_type]

    eye_tracking_01_filtered = filter_eye_tracking_data(eye_tracking_01, questions_01)
    eye_tracking_02_filtered = filter_eye_tracking_data(eye_tracking_02, questions_02)
    eye_tracking_03_filtered = filter_eye_tracking_data(eye_tracking_03, questions_03)

    metrics_01 = calculate_eye_tracking_metrics(eye_tracking_01_filtered)
    metrics_02 = calculate_eye_tracking_metrics(eye_tracking_02_filtered)
    metrics_03 = calculate_eye_tracking_metrics(eye_tracking_03_filtered)

    results.append({'Test': 'Test 01', 'Type': question_type, 'Start Time': questions_01['Question Start Time'].min().strftime('%H:%M:%S'), 'End Time': questions_01['Question Answer Time'].max().strftime('%H:%M:%S'),
                    'Average Pupil Dilation': metrics_01[0], 'Left Blink Rate': metrics_01[1], 'Right Blink Rate': metrics_01[2]})

    results.append({'Test': 'Test 02', 'Type': question_type, 'Start Time': questions_02['Question Start Time'].min().strftime('%H:%M:%S'), 'End Time': questions_02['Question Answer Time'].max().strftime('%H:%M:%S'),
                    'Average Pupil Dilation': metrics_02[0], 'Left Blink Rate': metrics_02[1], 'Right Blink Rate': metrics_02[2]})

    results.append({'Test': 'Test 03', 'Type': question_type, 'Start Time': questions_03['Question Start Time'].min().strftime('%H:%M:%S'), 'End Time': questions_03['Question Answer Time'].max().strftime('%H:%M:%S'),
                    'Average Pupil Dilation': metrics_03[0], 'Left Blink Rate': metrics_03[1], 'Right Blink Rate': metrics_03[2]})

# Show results
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))